# Credit Risk Modeling — Model Comparison

## Purpose of This Notebook

This notebook compares multiple supervised learning models for credit default prediction.

The goal is to evaluate:
- Predictive performance (ROC-AUC, Recall, Precision)
- Stability under class imbalance
- Interpretability vs performance trade-offs

Models compared:
- Logistic Regression (Baseline)
- Decision Tree
- Random Forest
- Gradient Boosting (XGBoost / GradientBoostingClassifier)

This notebook focuses on **model selection**, not deployment.


### Import Libraries

In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, classification_report)
import joblib

## Load Feature-Engineered Dataset

We use the dataset containing engineered behavioral, utilization, and delinquency features.

In [4]:
df=pd.read_excel("default of credit card clients_final.xlsx", header=0)
df.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,avg_utilization,max_utilization,payment_to_bill_ratio,payment_std,Avg_payment,bill_std,Avg_bill,high_utilization_flag,chronic_delinquency_flag,severe_delinquency_flag
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0.064200,0.195650,0.089434,281.283072,114.833333,1761.633219,1284.000000,0,0,0
1,120000,2,2,2,26,-1,2,0,0,0,...,0.023718,0.028792,0.292791,752.772653,833.333333,637.967841,2846.166667,0,0,0
2,90000,2,2,2,34,0,0,0,0,0,...,0.188246,0.324878,0.108388,1569.815488,1836.333333,6064.518593,16942.166667,0,0,0
3,50000,2,2,1,37,0,0,0,0,0,...,0.771113,0.985820,0.036259,478.058155,1398.000000,10565.793518,38555.666667,1,0,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,0.364463,0.716700,0.540054,13786.230736,9841.500000,10668.590074,18223.166667,0,0,0


## Define Target and Model Features

In [5]:
target = 'default payment next month'

features = ['Average_Delinquency','Max_Delinquency','number_of_delinquency_months','delinquency_trend',
'avg_utilization','max_utilization','payment_to_bill_ratio','payment_std','Avg_payment','bill_std','Avg_bill',
'high_utilization_flag','chronic_delinquency_flag','severe_delinquency_flag']

X = df[features]
y = df[target]


## Train–Test Split

A stratified split is used to preserve the default rate in both training and test sets.

In [7]:
X_train, X_test, Y_train, Y_test= train_test_split(X,y, test_size=0.25, stratify=y, random_state=42)

## Feature Scaling

Scaling is applied for linear models.
Tree-based models are scale-invariant but use the same input for consistency.

In [8]:
scaler=StandardScaler()
X_train_scaled= scaler.fit_transform(X_train)
X_test_scaled= scaler.transform(X_test)

## Model Definitions

Multiple models are defined to compare:
- Linear baseline
- Non-linear decision boundaries
- Ensemble learning

In [9]:
models={"Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000),
        "Decision Tree": DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', random_state=42),
        "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42)}

### Model Training & Evaluation Loop

In [24]:
results=[]
trained_models = {}
for name, model in models.items():
  if name=="Logistic Regression":
    model.fit(X_train_scaled, Y_train)
    y_pred_proba=model.predict_proba(X_test_scaled)[:,1]
    y_pred=model.predict(X_test_scaled)
  else:
      model.fit(X_train, Y_train)
      y_pred_proba=model.predict_proba(X_test)[:,1]
      y_pred= model.predict(X_test)
      roc= roc_auc_score(Y_test, y_pred_proba)
      results.append({"Model": name,
                "ROC_AUC": roc})
      trained_models[name] = model
      print(f"\n{name}")
      print(classification_report(Y_test, y_pred))


Decision Tree
              precision    recall  f1-score   support

           0       0.88      0.76      0.81      5841
           1       0.43      0.65      0.52      1659

    accuracy                           0.73      7500
   macro avg       0.66      0.70      0.67      7500
weighted avg       0.78      0.73      0.75      7500


Random Forest
              precision    recall  f1-score   support

           0       0.88      0.80      0.84      5841
           1       0.46      0.62      0.53      1659

    accuracy                           0.76      7500
   macro avg       0.67      0.71      0.68      7500
weighted avg       0.79      0.76      0.77      7500


Gradient Boosting
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      5841
           1       0.64      0.38      0.47      1659

    accuracy                           0.81      7500
   macro avg       0.74      0.66      0.68      7500
weighted avg       0.80  

Model Comparison — Performance Interpretation

Three tree-based classifiers were evaluated using class-balanced training to address the inherent class imbalance in credit default data. Model performance was assessed using ROC-AUC, precision/recall, and F1-score, with particular emphasis on the default (1) class, as this represents financial risk.

#### Decision Tree (Baseline Non-Linear Model)

- ROC-AUC: 0.760

- Default Recall: 0.65

- Default Precision: 0.43

#### Interpretation
The Decision Tree captures non-linear patterns and achieves relatively high recall for defaulters, meaning it flags a larger share of risky customers. However, this comes at the cost of low precision, resulting in many false positives.

#### Business Implication
Useful as a high-sensitivity screening model, but unsuitable as a standalone production model due to excessive unnecessary interventions.

#### Random Forest (Stability & Generalization)

- ROC-AUC: 0.775

- Default Recall: 0.62

- Default Precision: 0.46

#### Interpretation
Random Forest improves overall stability and balances precision and recall better than a single tree. It captures complex interactions while reducing variance.

#### Business Implication
A strong balanced-risk model suitable for environments where false positives are costly but missed defaults must still be controlled.

#### Gradient Boosting (Best Rank-Ordering Power)

- ROC-AUC: 0.776 (Best)

- Default Recall: 0.38

- Default Precision: 0.64

#### Interpretation
Gradient Boosting provides the strongest ability to rank customers by default risk, as reflected in the highest ROC-AUC. It is conservative in flagging defaults, prioritizing precision over recall.

#### Business Implication
Ideal for targeted, high-confidence interventions, such as proactive outreach or credit limit adjustments for the highest-risk customers.


## Model Performance Summary

In [19]:
results_df = pd.DataFrame(results).sort_values(by="ROC_AUC", ascending=False)
results_df

,Model,ROC_AUC
2,Gradient Boosting,0.776530
1,Random Forest,0.775192
0,Decision Tree,0.760372


In [25]:
joblib.dump(trained_models["Gradient Boosting"], "gradient_boosting_model.pkl")

['gradient_boosting_model.pkl']

Gradient Boosting is selected as the primary model due to its superior ROC-AUC and strong precision for default prediction. This makes it best suited for risk prioritization and early warning systems, where accurate ranking is more important than flagging every potential defaulter.

However:

- Random Forest is a strong secondary candidate when a more balanced recall-precision tradeoff is required.

- Decision Tree serves as an interpretable baseline but is not recommended for production use.

### Strategic Next Step

Model probabilities should now be calibrated and converted into business-aligned decision thresholds to support:

- Risk tiering (Low / Medium / High)

- Cost-sensitive intervention strategies

- Loss-minimization policies